In [1]:
!pip install accelerate

In [2]:
!pip install fastapi uvicorn pyngrok nest-asyncio python-multipart
# Cài đặt thêm các thư viện cho model Wan (giả định dùng diffusers hoặc repo gốc)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install git+https://github.com/huggingface/diffusers.git

Looking in indexes: https://download.pytorch.org/whl/cu121
  Cloning https://github.com/huggingface/diffusers.git to /tmp/pip-req-build-5cachk_1
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers.git /tmp/pip-req-build-5cachk_1
  Resolved https://github.com/huggingface/diffusers.git to commit 2ff5e5877f743b799cc389d7b60a054ce3b9ae50
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 4.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 38.2 MB/s eta 0:00:0000:0100:01
  Created wheel for diffusers: filename=diffusers-0.41.0.dev0-py3-none-any.whl size=5964566 sha256=4e51a608c43919bfa7c379a1b189d415bd9e72dbb36cc34

In [3]:
!pip install -U bitsandbytes transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 40.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 44.0 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
# 1. Gỡ sạch sẽ các bản bitsandbytes bị lỗi/cũ
!pip uninstall -y bitsandbytes

# 2. Cài đặt cứng phiên bản 0.47.0 (Bản ổn định nhất, không bị lỗi trên Kaggle)
!pip install bitsandbytes==0.47.0 accelerate transformers

# 3. Ép máy ảo khởi động lại hoàn toàn bằng code (Cách này mạnh hơn nút bấm Restart)
import os
os._exit(00)

Found existing installation: bitsandbytes 0.50.2
Uninstalling bitsandbytes-0.50.2:
  Successfully uninstalled bitsandbytes-0.50.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 26.9 MB/s eta 0:00:0000:01m0:01m


In [1]:
import bitsandbytes as bnb
import accelerate
import transformers

print("BitsAndBytes Version:", bnb.__version__)
print("Accelerate Version:", accelerate.__version__)
print("Transformers Version:", transformers.__version__)

BitsAndBytes Version: 0.47.0
Accelerate Version: 1.14.0
Transformers Version: 5.16.1


In [2]:
import torch
from huggingface_hub import login
# Sửa ở dòng này: Import đích danh LTXImageToVideoPipeline thay vì DiffusionPipeline chung chung
from diffusers import LTXImageToVideoPipeline 
from diffusers.utils import load_image, export_to_video

# 1. XÁC THỰC HUGGING FACE TOKEN
HF_TOKEN = "your_hf_token"
login(token=HF_TOKEN)

# 2. KHỞI TẠO MÔ HÌNH IMAGE-TO-VIDEO
print("Đang tải LTX-Video Pipeline...")
model_id = "Lightricks/LTX-Video" 

# Sử dụng LTXImageToVideoPipeline tại đây
pipe = LTXImageToVideoPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16, 
)

# 3. CÁC BƯỚC TỐI ƯU HÓA CHO MÔI TRƯỜNG KAGGLE
pipe.enable_model_cpu_offload() 
pipe.vae.enable_tiling()

print("✅ Đã tải và tối ưu xong mô hình! Sẵn sàng tích hợp vào FastAPI.")

Đang tải LTX-Video Pipeline...


/usr/local/lib/python3.12/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

✅ Đã tải và tối ưu xong mô hình! Sẵn sàng tích hợp vào FastAPI.


In [2]:
# ==========================================
# 4. CHẠY THỬ (Bỏ comment phần dưới để test trước khi lên API)
# ==========================================

# prompt = "A man with short gray hair plays a red electric guitar."
# image = load_image("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/guitar-man.png")

# print("Đang generate video từ Text + Image...")

# # Dòng này chính là nơi model nhận CẢ văn bản lẫn hình ảnh đầu vào:
# output = pipe(
#     image=image,   
#     prompt=prompt, 
#     height=288,
#     width=512,
#     num_frames=17,
#     num_inference_steps=20
#     # ĐÃ XÓA decode_chunk_size ở đây!
# ).frames[0]

# export_to_video(output, "output.mp4")
# print("Hoàn thành! Video đã được lưu thành output.mp4")

Đang generate video từ Text + Image...


KeyboardInterrupt: 

**Chỉ sử dụng nếu cần clear cache**

In [ ]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

### 3. Setup Ngrok & FastAPI

In [3]:
from pyngrok import ngrok
import uvicorn
import threading
import nest_asyncio

# Khắc phục lỗi event loop khi chạy Uvicorn trong Jupyter Notebook
nest_asyncio.apply()

# Đóng các tunnel cũ để tránh xung đột port
ngrok.kill()

# Điền Token của bạn (⚠️ LƯU Ý: Bạn lại mới để lộ token Ngrok trong notebook rồi kìa! Nhớ vào dashboard ngrok reset lại token đi nhé để tránh bị người khác dùng trộm)
NGROK_AUTH_TOKEN = "your_ngrok_token"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Mở kết nối
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url


In [4]:
from fastapi import FastAPI, File, UploadFile, Form, BackgroundTasks
from fastapi.responses import FileResponse, JSONResponse
from diffusers.utils import export_to_video
import io
import os
import uuid
import torch
from PIL import Image

app = FastAPI(title="LTX Image-to-Video API")

# Dictionary lưu trạng thái các video đang render
task_status = {}

@app.get("/")
def read_root():
    return {"message": "LTX-Video API đang hoạt động! Trạng thái: Sẵn sàng."}

@app.get("/videos/{filename}")
async def get_video(filename: str):
    file_path = f"/kaggle/working/{filename}"
    if os.path.exists(file_path):
        return FileResponse(file_path, media_type="video/mp4")
    return JSONResponse(status_code=404, content={"message": "Video không tồn tại hoặc chưa render xong!"})

# 2. API Tạo video
# @app.post("/generate-video")
# async def generate_video(background_tasks: BackgroundTasks, image: UploadFile = File(...), prompt: str = Form(...)):
#     task_id = str(uuid.uuid4())[:8]
#     task_status[task_id] = {"status": "processing"}
    
#     clean_name = image.filename.split('.')[0]
#     video_filename = f"video_{clean_name}_{task_id}.mp4"
    
#     # Đọc file ảnh
#     image_bytes = await image.read()
    
#     # Đưa việc render vào luồng ngầm
#     background_tasks.add_task(process_video_bg, task_id, image_bytes, video_filename, prompt)
    
#     return {
#         "message": "Đang render...",
#         "task_id": task_id,
#         "check_status_endpoint": f"/status/{task_id}",
#         "video_endpoint": f"/videos/{video_filename}"
#     }

@app.post("/generate-video")
async def generate_video(
    background_tasks: BackgroundTasks, 
    image: UploadFile = File(...), 
    prompt: str = Form(...),
    ratio: str = Form("16:9") # Thêm tham số chọn tỷ lệ từ người dùng
):
    task_id = str(uuid.uuid4())[:8]
    task_status[task_id] = {"status": "processing"}
    
    clean_name = image.filename.split('.')[0]
    video_filename = f"video_{clean_name}_{task_id}.mp4"
    
    # Đọc file ảnh
    image_bytes = await image.read()

    # --- ĐOẠN MỚI: Xử lý quy đổi Tỷ lệ (Ratio) sang Kích thước (Width/Height) ---
    if ratio == "16:9":     # Video ngang (Youtube, Web)
        w, h = 1024, 576
    elif ratio == "9:16":   # Video dọc (TikTok, Reels)
        w, h = 576, 1024
    elif ratio == "1:1":    # Video vuông (Instagram)
        w, h = 768, 768 
    elif ratio == "21:9":   # Video Cinematic (Siêu rộng)
        w, h = 1024, 448
    else:
        w, h = 1024, 576    # Fallback mặc định nếu người dùng nhập linh tinh
    # ------------------------------------------------------------------------

    # Đưa task vào chạy ngầm, NHỚ truyền thêm w và h vào hàm process_video_bg
    background_tasks.add_task(
        process_video_bg, 
        task_id, 
        image_bytes, 
        video_filename, 
        prompt, 
        w, 
        h
    )
    
    return {
        "message": "Đang xử lý video", 
        "task_id": task_id, 
        "resolution": f"{w}x{h}",
        "check_status_url": f"{public_url}/status/{task_id}",
        "video_url": f"{public_url}/videos/{video_filename}"
    }

# API Kiểm tra trạng thái
@app.get("/status/{task_id}")
def check_status(task_id: str):
    status = task_status.get(task_id)
    if not status:
        return JSONResponse(status_code=404, content={"message": "Không tìm thấy task ID này."})
    return status

In [6]:
import subprocess
import os
import io
from PIL import Image
from diffusers.utils import export_to_video

# 1. THÊM width: int, height: int VÀO ĐÂY
def process_video_bg(task_id: str, image_bytes: bytes, video_filename: str, prompt: str, width: int, height: int):
    try:
        init_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        print(f"[*] Task {task_id}: Đang render với prompt: '{prompt}' ở độ phân giải {width}x{height}...")
        
        # 2. RESIZE ẢNH ĐẦU VÀO CHO KHỚP VỚI KHUNG HÌNH (Rất quan trọng để không lỗi model)
        init_image = init_image.resize((width, height), Image.LANCZOS)

        # 3. TRUYỀN BIẾN width VÀ height VÀO MODEL THAY VÌ SỐ CỨNG
        frames = pipe(
            image=init_image, 
            prompt=prompt, 
            height=height,    # Đã sửa thành biến động
            width=width,      # Đã sửa thành biến động
            num_frames=41,    # công thức là (8 * n) + 1 
            num_inference_steps=20
        ).frames[0]
        
        temp_output = f"/kaggle/working/temp_{video_filename}"
        final_output = f"/kaggle/working/{video_filename}"
        
        export_to_video(frames, temp_output, fps=8)
        print(f"[*] Task {task_id}: Đang convert sang chuẩn H.264 cho Web...")
        
        subprocess.run(["ffmpeg", "-y", "-i", temp_output, "-vcodec", "libx264", "-pix_fmt", "yuv420p", final_output], 
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        if os.path.exists(temp_output):
            os.remove(temp_output)
            
        print(f"[*] Task {task_id}: Render và Convert thành công!")
        
        # Nhớ giữ nguyên video_url bạn vừa đổi ban nãy nhé
        task_status[task_id] = {
            "status": "completed", 
            "video_filename": video_filename,
            "video_url": f"{public_url}/videos/{video_filename}", 
            "prompt": prompt
        }

    except Exception as e:
        print(f"[!] Task {task_id} Lỗi: {e}")
        task_status[task_id] = {"status": "error", "message": str(e)}

In [8]:

print("=" * 60)
print(f"🚀 SERVER API ĐÃ ONLINE TẠI: {public_url}")
print(f"👉 Endpoint tạo video: POST {public_url}/generate-video")
print(f"👉 Tài liệu Test API trực quan: {public_url}/docs")
print("=" * 60)

# Kỹ thuật chạy Server ngầm (Background Thread)
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server)
thread.daemon = True
thread.start()

print("-> [OK] Server đang lắng nghe ở chế độ nền. Bác có thể test API được rồi!")

🚀 SERVER API ĐÃ ONLINE TẠI: https://defection-rimless-bobble.ngrok-free.dev
👉 Endpoint tạo video: POST https://defection-rimless-bobble.ngrok-free.dev/generate-video
👉 Tài liệu Test API trực quan: https://defection-rimless-bobble.ngrok-free.dev/docs
-> [OK] Server đang lắng nghe ở chế độ nền. Bác có thể test API được rồi!


INFO:     Started server process [180]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     14.241.228.11:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/8377652c HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
[*] Task 1b9f0617: Đang convert sang chuẩn H.264 cho Web...
[*] Task 1b9f0617: Render và Convert thành công!
INFO:     14.241.228.11:0 - "POST /generate-video HTTP/1.1" 200 OK
[*] Task 27941c6a: Đang render với prompt: 'A beautiful Asian woman in a red lace top and a flowing black skirt dancing passionately in a brightly lit dance studio. She gracefully and dynamically twirls the edges of her black skirt, stepping rhythmically. The fabric of the skirt billows heavily. Smooth, fluid motion, highly detailed, realistic, 4k.' ở độ phân giải 576x1024...


  0%|          | 0/20 [00:00<?, ?it/s]

INFO:     14.241.228.11:0 - "GET /status/1b9f0617 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_6741245785261297897_1b9f0617.mp4 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_6741245785261297897_1b9f0617.mp4 HTTP/1.1" 206 Partial Content
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_6741245785261297897_1b9f0617.mp4 HTTP/1.1" 206 Partial Content
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_6741245785261297897_1b9f0617.mp4 HTTP/1.1" 206 Partial Content
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_6741245785261297897_1b9f0617.mp4 HTTP/1.1" 206 Partial Content
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_6741245785261297897_1b9f0617.mp4 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_6741245785261297897_1b9f0617.mp4 HTTP/1.1" 206 Partial Content
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_6741245785261297897_1b9f0617.mp4 HTTP/1.1" 206 Partial

...

INFO:     14.241.228.11:0 - "GET /status/e1ac9721 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/e1ac9721 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/e1ac9721 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/e1ac9721 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/e1ac9721 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/e1ac9721 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/e1ac9721 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/e1ac9721 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/e1ac9721 HTTP/1.1" 200 OK
INFO:     14.241.228.13:0 - "GET /status/e1ac9721 HTTP/1.1" 200 OK
INFO:     14.241.228.13:0 - "GET /status/e1ac9721 HTTP/1.1" 200 OK
INFO:     14.241.228.13:0 - "GET /status/e1ac9721 HTTP/1.1" 200 OK
INFO:     14.241.228.13:0 - "GET /status/e1ac9721 HTTP/1.1" 200 OK
INFO:     14.241.228.13:0 - "GET /status/e1ac9721 HTTP/1.1" 200 OK
INFO:     14.241.228.13:0 - "GET /status/e1ac9721 HTTP/1.1" 20

  0%|          | 0/20 [00:00<?, ?it/s]

INFO:     14.241.228.13:0 - "GET /status/dfc713a8 HTTP/1.1" 200 OK
INFO:     14.241.228.13:0 - "GET /status/dfc713a8 HTTP/1.1" 200 OK
INFO:     14.241.228.13:0 - "GET /status/dfc713a8 HTTP/1.1" 200 OK
INFO:     14.241.228.13:0 - "GET /status/dfc713a8 HTTP/1.1" 200 OK
INFO:     14.241.228.13:0 - "GET /status/dfc713a8 HTTP/1.1" 200 OK
INFO:     14.241.228.13:0 - "GET /status/dfc713a8 HTTP/1.1" 200 OK
INFO:     14.241.228.13:0 - "GET /status/dfc713a8 HTTP/1.1" 200 OK
INFO:     14.241.228.13:0 - "GET /status/dfc713a8 HTTP/1.1" 200 OK
INFO:     14.241.228.13:0 - "GET /status/dfc713a8 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/dfc713a8 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/dfc713a8 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/dfc713a8 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/dfc713a8 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/dfc713a8 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/dfc713a8 HTTP/1.1" 20

  0%|          | 0/20 [00:00<?, ?it/s]

INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /status/ddf4738a HTTP/1.1" 20

In [ ]:
import time
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    print("Dừng server.")

INFO:     14.241.228.13:0 - "GET /status/fc5dfa69 HTTP/1.1" 200 OK
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_1830867471711778971_fc5dfa69.mp4 HTTP/1.1" 404 Not Found
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_1830867471711778971_fc5dfa69.mp4 HTTP/1.1" 404 Not Found
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_1830867471711778971_fc5dfa69.mp4 HTTP/1.1" 404 Not Found
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_1830867471711778971_fc5dfa69.mp4 HTTP/1.1" 404 Not Found
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_1830867471711778971_fc5dfa69.mp4 HTTP/1.1" 404 Not Found
INFO:     14.241.228.13:0 - "GET /videos/video_upload_img_1830867471711778971_fc5dfa69.mp4 HTTP/1.1" 404 Not Found
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_1830867471711778971_fc5dfa69.mp4 HTTP/1.1" 404 Not Found
INFO:     14.241.228.11:0 - "GET /videos/video_upload_img_1830867471711778971_fc5dfa69.mp4 HTTP/1.1" 404 Not Found
INFO:     14.